##### 샘플 데이터

In [ ]:
import pandas as pd
import re
from tqdm import tqdm
from konlpy.tag import Okt
import pickle
import gensim #자연어 처리 및 토픽 모델링을 위한 라이브러리
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
import random

In [ ]:
df = pd.read_csv("ALL_DATA_MERGED_FINAL.csv", encoding="utf-8")

In [ ]:
def re_pattern(string) :
    pattern = re.compile(r'[^a-zA-Z가-힣\s\.\?\!]')
    string = re.sub(pattern, ' ', string)

    pattern2 = re.compile(r'\s+')
    result = re.sub(pattern2, ' ', string)
    return result

In [ ]:
# .copy()를 추가하여 독립된 데이터프레임으로 생성
df = df.dropna(subset=['content']).copy()

df['content'] = df['content'].apply(lambda x : re_pattern(x))

In [ ]:
# apply
df = df[df['content'].apply(lambda x : len(x) > 15)]
# df

In [ ]:
df = df.reset_index(drop=True)
len(df)

In [ ]:
# 추가 전처리

df['content'] = df['content'].str.strip()  # 앞뒤 공백 제거
df['content'] = df['content'].str.replace("\n", " ")  # 줄바꿈 제거


In [ ]:
# content 컬럼 기준으로 중복 행 제거
df = df.drop_duplicates(subset=['content'], keep='first')
df = df.reset_index(drop=True)
len(df)

In [ ]:
okt=Okt()
stopwords_df = pd.read_csv(r"ko-stopwords.csv")

In [ ]:
stopwords = set(stopwords_df['stopwords']) #집합 변경

In [ ]:
def okt_pos_tagging(string) :
    pos_words = okt.pos(string, stem =True, norm=True)
    result = [word for word, tag in pos_words if word not in stopwords if tag in {'Noun', 'Adjective', 'Verb'} ]
    return result

In [ ]:
tqdm.pandas()

In [ ]:
df['tagged_content'] = df['content'].progress_apply(lambda x : okt_pos_tagging(x))

In [ ]:
with open("tagged_content_1119.pkl", 'wb') as f:
    pickle.dump(df, f)

In [ ]:
with open("tagged_content_1119.pkl", "rb") as f:
    data = pickle.load(f)

df = pd.DataFrame(data)

In [ ]:
tagged_corpus_list = []

for n, i in enumerate(df['tagged_content']):
    tag = "document{}".format(n) # ID 만들기
    tagged_corpus_list.append(TaggedDocument(tags=[tag], words = i))

In [ ]:
model_doc2vec = Doc2Vec(vector_size = 200,
                        alpha = 0.025,
                        min_alpha = 0.01,
                        window = 3,
                        min_count = 5,
                        dm = 0
                       )

In [ ]:
#단어사전 장착
model_doc2vec.build_vocab(tagged_corpus_list)

In [ ]:
n_epochs = 20

for epoch in tqdm(range(n_epochs), desc="Training Epochs"):
    random.shuffle(tagged_corpus_list)  # 매 epoch마다 데이터 섞기 추천
    model_doc2vec.train(
        tagged_corpus_list,
        total_examples=model_doc2vec.corpus_count,
        epochs=1
    )


In [ ]:
vector_list = []
for i in range(len(df)) :
    doc2vec = model_doc2vec.dv["document{}".format(i)]
    vector_list.append(doc2vec)

In [ ]:
df['vector'] = vector_list

In [ ]:
# 저장하기
import pickle
with open('1119_vector_df_2.pkl', 'wb') as f:
    pickle.dump(df, f)

In [ ]:
# # 불러오기
import pickle
with open('1119_vector_df_2.pkl', 'rb') as f:
    df = pickle.load(f)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from matplotlib import pyplot as plt
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

In [ ]:
####### 이슈 log : 데이터 많아서 memory error 발생
# 샘플링해서 일부만 그려보기 : 무작위 추출
 
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
 
# 데이터 불러오기 (리스트 형태 → 2D 배열)
X = np.vstack(df['vector'].to_list()).astype('float32')
 
# 2000개 랜덤 샘플
n = min(3000, len(X))
idx = np.random.choice(len(X), size=n, replace=False)
X_sample = X[idx]
 
# 계층적 군집 + 덴드로그램
Z = linkage(X_sample, method='ward')
plt.figure(figsize=(12, 6))
dendrogram(Z, truncate_mode="level", p=10)  # 가지 너무 많을 때 잘라서 보기
plt.show()

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.metrics.cluster import silhouette_score
from sklearn.cluster import AgglomerativeClustering

In [ ]:
# ############## 기본 코드:  n개의 클러스터로 구분해 보기 #################
# cluster_model = AgglomerativeClustering(n_clusters=5, linkage='ward')
# cluster_label = cluster_model.fit_predict(list(df['vector']))
# cluster_label
# ###############################################################

import time
import numpy as np
from sklearn.cluster import AgglomerativeClustering

print(">>> Clustering start (n_clusters=5, linkage='ward')", flush=True)
t0 = time.time()

cluster_model = AgglomerativeClustering(n_clusters=5, linkage='ward')
cluster_label = cluster_model.fit_predict(list(df['vector']))  # 기존 코드 유지

elapsed = time.time() - t0
print(f">>> Done in {elapsed:.2f} sec", flush=True)

# 결과 요약
print(">>> Unique labels:", np.unique(cluster_label))
print(">>> Counts by label:", np.bincount(cluster_label))


In [ ]:
df

In [ ]:
df['cluster'] = cluster_label

In [ ]:
# 피클 파일로 저장하기
import pickle

with open("cluster_label_0924.pkl", 'wb') as f:
    pickle.dump(df, f)

In [ ]:
df2 = df.sample(frac=1).reset_index(drop=True)
df2 =  df2[:3000]
df2

In [ ]:
# 실루엣 지수 구하기
n_cluster = [
]
clustering_score = []

for i in tqdm(range(2,15)) :
    cluster_model = AgglomerativeClustering(n_clusters=i , linkage='ward')
    cluster_label = cluster_model.fit_predict(list(df2['vector']))
    score = silhouette_score(list(df2['vector']), cluster_label)
    n_cluster.append(i)
    clustering_score.append(score)

In [ ]:
import numpy as np
from tqdm import tqdm
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# 샘플링
df2 = df.sample(frac=1, random_state=42).reset_index(drop=True).iloc[:3000]

# numpy 배열로 변환
X = np.vstack(df2['vector'].to_numpy())

n_cluster = []
clustering_score = []

for i in tqdm(range(2, 15)):
    cluster_model = AgglomerativeClustering(n_clusters=i, linkage='ward')
    cluster_label = cluster_model.fit_predict(X)
    score = silhouette_score(X, cluster_label)
    n_cluster.append(i)
    clustering_score.append(score)

print(list(zip(n_cluster, clustering_score)))


In [ ]:
clustering_score

In [ ]:
plt.plot(n_cluster, clustering_score)

In [ ]:
result = pd.DataFrame({'n_cluster':n_cluster, 'score':clustering_score})
result

In [ ]:
from gensim.models import CoherenceModel #coherence 모델 라이브러리
import matplotlib.pyplot as plt #그래프 그리기
import numpy as np
import gensim
from gensim import corpora, models
from gensim.corpora import Dictionary

In [ ]:
import gensim
from gensim import corpora, models

In [ ]:
dictionary = corpora.Dictionary(df['tagged_content'])

In [ ]:
# ID랑 매칭
corpus = [dictionary.doc2bow(doc) for doc in df['tagged_content']]
corpus

In [ ]:
top_n = 3
# 모델 생성
ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics=top_n, id2word = dictionary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import gensim
from gensim.models import CoherenceModel

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# 1. 데이터 준비 (가장 중요: 벡터 고정)
# 리스트 형태의 벡터를 2차원 numpy 배열로 변환
X_fixed = np.vstack(df['vector'].to_list())

# 결과 저장 리스트
inertia_scores = []   # Elbow (낮을수록 좋음, 꺾이는 지점 확인)
silhouette_scores = [] # Silhouette (높을수록 좋음)

# 군집 개수 범위 (2 ~ 9)
k_range = range(2, 10)

print("K-Means & Silhouette 분석 시작...")

for k in k_range:
    # 모델 생성 및 학습
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_fixed)
    
    # 1. Elbow (Inertia) 저장
    inertia_scores.append(kmeans.inertia_)
    
    # 2. Silhouette Score 저장
    score = silhouette_score(X_fixed, labels)
    silhouette_scores.append(score)
    
    print(f"K={k} 완료 | Silhouette: {score:.4f}")

# --- 시각화 ---
plt.figure(figsize=(15, 6))

# 왼쪽 그래프: Silhouette Score
plt.subplot(1, 2, 1)
plt.plot(k_range, silhouette_scores, 'o-', color='blue')
plt.title('Silhouette Score (Higher is Better)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.grid(True)

# 오른쪽 그래프: Elbow Method (Inertia)
plt.subplot(1, 2, 2)
plt.plot(k_range, inertia_scores, 'o-', color='red')
plt.title('Elbow Method (Lower is Better)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
x = range(2,10)

plt.plot(x, c_score)
plt.show()

In [ ]:
x = range(2,10)

plt.plot(x,p_score)
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering

In [ ]:
cluster_model = AgglomerativeClustering(n_clusters=3, linkage='ward')
cluster_label = cluster_model.fit_predict(list(df['vector']))
cluster_label

In [ ]:
df['cluster'] = cluster_label

In [ ]:
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
all_document = []

for i in df['cluster'].unique() :
    print(i)
    pos_tagging = df[df['cluster'] == i]['tagged_content'] 

    document = ''
    for pos in pos_tagging :
        doc = ' '.join(pos) + ' '
        document += doc
    all_document.append(document)

In [ ]:
#TF-IDF 벡터라이저 선언
vectorizer = TfidfVectorizer()
# TF-IDF 메트릭스 핏
tfidf_matrix = vectorizer.fit_transform(all_document)

In [ ]:
# 키워드도출
feature_name = vectorizer.get_feature_names_out()

In [ ]:
# TF-IDF 값
tfidf_value = tfidf_matrix.toarray()

In [ ]:
tfidf_df = pd.DataFrame(tfidf_value, columns = feature_name)

In [ ]:
tfidf_df.index = df['cluster'].unique()

In [ ]:
tfidf_df_T = tfidf_df.T

In [ ]:
for i in tfidf_df_T.columns :
    #점수 정렬
    tfidfvalue = tfidf_df_T[i].sort_values(ascending=False)
    #데이터 프레임으로 변환
    data = {'tfidf_word': tfidfvalue.index, 'tfidf': tfidfvalue.values}
    data_df = pd.DataFrame(data)
    data_df.to_csv(f'cluster{i}_tf_idf_0924_ 2차(5로).csv', encoding='utf-8-sig')

In [ ]:
import pickle

with open('clustering_result_1119_2차(5로).pkl', 'wb') as f:
    pickle.dump(df, f)

In [ ]:
df.to_csv("Actor 클러스터링.csv",encoding = 'utf-8-sig')